# ASTER — Distinguer adaptation et résolution du localisateur

**Notebook Colab à exécuter de haut en bas sur un GPU.** Il reprend le protocole
`05_mixedreplay_x40_colab.ipynb` pour la base retenue **YOLO11n exp4/nonone**.
Il produit de nouvelles mesures, sans présumer retrouver 0,371 ou 0,982.

**Question :** quelle amélioration subsiste à résolution de test identique, et
l'adaptation reste-t-elle efficace lorsqu'elle est réalisée à 640 pixels ?

| Modèle | Apprentissage | Test 640 | Test 960 |
|---|---|---|---|
| Initial | Sources publiques à 640 | Oui | Oui |
| Adapté 640 | Même modèle initial, adaptation mixed-replay à 640 | Oui | Oui |
| Adapté 960 | Même modèle initial, adaptation mixed-replay à 960 | Oui | Oui |

Deux modes :
- **`economique` (par défaut)** : un nouvel entraînement à 640 ; le modèle adapté à 960
  vient du Drive. Répond au contrôle pratique demandé. La comparaison des deux
  entraînements reste historique : environnement et tirage de replay anciens non certifiés.
- **`controle` (comparaison plus rigoureuse)** : deux nouveaux entraînements, 640 et 960,
  même base, mêmes listes, seed, batch, augmentations et environnement. La résolution
  est le seul paramètre d'entraînement explicitement modifié.

Le test contient les **104 champs et 171 boîtes** du découpage existant. On ne change
ni ce découpage ni le modèle de départ selon les scores obtenus. Les modèles sont
choisis sur la validation, avant les tests. Le domaine public est également évalué.

**Statut livré :** sans résultats expérimentaux ; exécution GPU/Drive à effectuer sur Colab.

## 1. Fichiers requis et réglages

Dans Colab : **Exécution → Modifier le type d'exécution → GPU**, puis **Tout exécuter**.
Les chemins par défaut reprennent ceux du notebook d'origine :

```text
MyDrive/LLD/
  lld_cloud.tar.part-*                 archive source découpée
  state/runs/exp4_domain_aug__nonone/weights/best.pt
  state/runs/mixedreplay__exp4_domain_aug__nonone/weights/best.pt  (mode economique)
MyDrive/aster_x40dataset/
  images/{train,val,test}/*.jpg
  labels/{train,val,test}/*.txt
```

Si les poids adaptés sont uniquement sur ton Mac, copie **`wbc_detector.pt`** depuis
`ASTER_deployed_repo_complete/models/yolo/` vers ton Drive et renseigne
`HISTORICAL_960_WEIGHTS` ci-dessous. Le modèle de base doit être celui **avant adaptation**.
Aucune copie automatique vers Drive n'est faite depuis le Mac par ce fichier.

Tous les nouveaux résultats iront dans **`MyDrive/LLD/resolution_ablation_v1/`**.
Aucun poids ou résultat historique dans `state/` n'est modifié.
Pour une comparaison contrôlée, choisir `MODE = "controle"` **avant** de tout exécuter.

In [ ]:
from pathlib import Path

MODE = "economique"                  # "economique" ou "controle"
DRIVE_LLD = Path("/content/drive/MyDrive/LLD")
DRIVE_X40 = Path("/content/drive/MyDrive/aster_x40dataset")
BASE_WEIGHTS = DRIVE_LLD / "state/runs/exp4_domain_aug__nonone/weights/best.pt"
HISTORICAL_960_WEIGHTS = DRIVE_LLD / "state/runs/mixedreplay__exp4_domain_aug__nonone/weights/best.pt"
OUTPUT_ROOT = DRIVE_LLD / "resolution_ablation_v1"

ULTRALYTICS_VERSION = "8.4.53"       # version déclarée dans l'article
TRAIN_BATCH = 16                     # commun aux deux résolutions ; pas de batch automatique
EPOCHS = 60
PATIENCE = 10
TRAIN_SEED = 0
REPLAY_SEED = 42
REPLAY_RATIO = 4
WORKERS = 8
LR0 = 0.001
TEST_SIZES = (640, 960)
EVAL_BATCH = 1                      # même batch pour toutes les évaluations
NMS_IOU = 0.50                      # même suppression des doublons pour tous les modèles
OPERATING_CONF = 0.18               # seuil historique, jamais réoptimisé sur le test
AUG = dict(hsv_h=0.5, hsv_s=0.7, hsv_v=0.4, scale=0.5,
           degrees=15.0, mosaic=1.0, fliplr=0.5, flipud=0.5)
assert MODE in {"economique", "controle"}

## 2. Installer et vérifier le GPU

La version d'Ultralytics est fixée. PyTorch/CUDA sont ceux du runtime Colab et seront
archivés avec les résultats. Si Colab demande un redémarrage après installation,
redémarrer puis reprendre depuis la première cellule. En cas de manque de mémoire,
choisir un GPU plus grand ou réduire **le même `TRAIN_BATCH` pour les deux résolutions**
puis relancer ; ne pas modifier un seul bras de l'expérience.

In [ ]:
import sys, subprocess, os, json, shutil, hashlib, random, time, csv, gc
from datetime import datetime, timezone
from google.colab import drive

drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"ultralytics=={ULTRALYTICS_VERSION}", "pandas", "matplotlib", "pyyaml"], check=True)
import torch, ultralytics, numpy as np, pandas as pd, yaml
from PIL import Image
from ultralytics import YOLO
from IPython.display import display
assert ultralytics.__version__ == ULTRALYTICS_VERSION, "Redémarrer le runtime après installation."
assert torch.cuda.is_available(), "Activer un runtime GPU dans Colab."
assert DRIVE_LLD.is_dir() and DRIVE_X40.is_dir(), "Corriger DRIVE_LLD / DRIVE_X40 dans la cellule 1."
assert BASE_WEIGHTS.is_file(), f"Poids initiaux manquants : {BASE_WEIGHTS}"
if MODE == "economique":
    assert HISTORICAL_960_WEIGHTS.is_file(), f"Poids adaptés 960 manquants : {HISTORICAL_960_WEIGHTS}"
DEVICE = "0"
ENV = dict(python=sys.version, ultralytics=ultralytics.__version__, torch=torch.__version__,
           cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
           gpu=torch.cuda.get_device_name(0), numpy=np.__version__, pandas=pd.__version__)
print(json.dumps(ENV, indent=2))

## 3. Préparer les données locales

Comme dans le notebook d'origine, les archives LLD sont extraites dans `/content/lld_cloud`
et le dataset prototype est copié sur le disque Colab. Une vue locale séparée associe
les images publiques aux annotations **nonone** ; les annotations originales sont conservées.

In [ ]:
ROOT = Path("/content/lld_cloud")
WORK = Path("/content/aster_resolution_v1")
X40 = WORK / "x40"
SOURCE_VIEW = WORK / "source"
WORK.mkdir(parents=True, exist_ok=True)

if not (ROOT / "data/images").is_dir():
    parts = sorted(DRIVE_LLD.glob("lld_cloud.tar.part-*"))
    assert parts, f"Archives lld_cloud.tar.part-* absentes de {DRIVE_LLD}"
    # Pipeline sans interpolation shell ; toute erreur d'extraction arrête le notebook.
    with subprocess.Popen(["cat", *map(str, parts)], stdout=subprocess.PIPE) as producer:
        extraction = subprocess.run(["tar", "--warning=no-unknown-keyword", "-xf", "-", "-C", "/content"],
                                    stdin=producer.stdout)
        producer.stdout.close()
        rc = producer.wait()
    assert rc == 0 and extraction.returncode == 0, "Extraction LLD incomplète."
assert (ROOT / "data/images").is_dir(), "Structure LLD attendue : lld_cloud/data/images."
assert (ROOT / "labels_variants/nonone").is_dir(), "Annotations nonone absentes."
shutil.copytree(DRIVE_X40, X40, dirs_exist_ok=True, ignore=shutil.ignore_patterns("._*", ".DS_Store", "*.cache"))
SOURCE_VIEW.mkdir(exist_ok=True)
for name, target in [("images", ROOT / "data/images"), ("labels", ROOT / "labels_variants/nonone")]:
    link = SOURCE_VIEW / name
    if link.is_symlink():
        assert link.resolve() == target.resolve(), f"Lien local inattendu : {link}"
    else:
        assert not link.exists(), f"Dossier local inattendu : {link}"
        link.symlink_to(target, target_is_directory=True)
print("Sources et prototype disponibles localement.")

## 4. Reprendre les partitions et le tirage de replay

Le tirage reprend l'ordre des listes originales, avec `random.Random(42)` puis le tirage
train et val successivement. Les deux adaptations utilisent les **mêmes fichiers YAML**.
La validation contient 43 champs prototype et le replay de validation source ; le test
n'est utilisé ni pour l'arrêt anticipé ni pour choisir entre les modèles.

In [ ]:
def source_list(split):
    path = ROOT / f"lists/exp4_domain_aug_{split}.txt"
    assert path.is_file(), f"Liste source absente : {path}"
    result = []
    for raw in path.read_text().splitlines():
        raw = raw.strip()
        if not raw:
            continue
        normalized = raw.replace("\\", "/")
        marker = "data/images/"
        assert marker in normalized, f"Chemin source non reconnu : {raw}"
        relative = normalized.split(marker, 1)[1]
        assert ".." not in Path(relative).parts, raw
        result.append(SOURCE_VIEW / "images" / relative)
    return result

splits = ("train", "val", "test")
x_lists = {s: sorted(p for p in (X40 / "images" / s).glob("*.jpg") if not p.name.startswith("._")) for s in splits}
s_lists = {s: source_list(s) for s in splits}
expected = {"train": (204, 334), "val": (43, 67), "test": (104, 171)}

def label_path(image_path):
    p = str(image_path)
    assert "/images/" in p
    return Path(p.replace("/images/", "/labels/", 1)).with_suffix(".txt")

for collection in (x_lists, s_lists):
    for split, paths in collection.items():
        assert paths and len(paths) == len(set(paths)), f"Liste vide ou doublons : {split}"
        assert all(p.is_file() for p in paths), f"Images manquantes : {split}"
        assert all(label_path(p).is_file() for p in paths), f"Annotations manquantes : {split}"
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        assert not (set(collection[a]) & set(collection[b])), f"Chevauchement de chemins : {a}/{b}"
# Les noms des images prototype encodent les captures ; aucun ne doit traverser les partitions.
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not ({p.name for p in x_lists[a]} & {p.name for p in x_lists[b]}), "Capture répétée entre partitions."

rng = random.Random(REPLAY_SEED)
replay_train = rng.sample(s_lists["train"], min(len(s_lists["train"]), REPLAY_RATIO * len(x_lists["train"])))
replay_val = rng.sample(s_lists["val"], min(len(s_lists["val"]), REPLAY_RATIO * len(x_lists["val"])))
used = {**{f"x40_{s}": x_lists[s] for s in splits}, "source_replay_train": replay_train,
        "source_replay_val": replay_val, "source_test": s_lists["test"]}
print(pd.DataFrame([{"partition": key, "images": len(paths)} for key, paths in used.items()]).to_string(index=False))

## 5. Contrôler les annotations et enregistrer les empreintes

Les empreintes SHA-256 identifient les images, annotations et poids effectivement utilisés.
Les doublons binaires entre les partitions utilisées sont refusés. Ce contrôle ne prouve
pas l'absence de champs visuellement proches ou de patients communs dans les sources publiques.
Les groupes temporels du découpage prototype original sont conservés, sans nouveau tirage.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False))
    tmp.replace(path)

def validate_label(path):
    lines = [line.split() for line in path.read_text().splitlines() if line.strip()]
    for line in lines:
        assert len(line) == 5, f"Annotation invalide : {path}"
        c, x, y, w, h = map(float, line)
        assert all(np.isfinite([c, x, y, w, h])), path
        assert c == 0 and 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1, path
    return len(lines)

manifest = []
seen_hashes = {}
for role, paths in used.items():
    count = 0
    for p in paths:
        label = label_path(p)
        count += validate_label(label)
        image_hash = sha256_file(p)
        partition = role.rsplit("_", 1)[-1]
        if image_hash in seen_hashes:
            assert seen_hashes[image_hash] == partition, f"Image identique entre partitions : {p}"
        seen_hashes[image_hash] = partition
        manifest.append(dict(role=role, image=str(p), image_sha256=image_hash,
                             label=str(label), label_sha256=sha256_file(label)))
    if role.startswith("x40_"):
        assert (len(paths), count) == expected[partition], f"Découpage prototype inattendu : {role}, {len(paths)} images, {count} boîtes"
    print(role, len(paths), "images /", count, "boîtes")

INPUTS = WORK / "weights_inputs"
INPUTS.mkdir(exist_ok=True)
BASE_LOCAL = INPUTS / "base_640.pt"
shutil.copy2(BASE_WEIGHTS, BASE_LOCAL)
HIST_LOCAL = INPUTS / "adapted_960_historical.pt"
if MODE == "economique":
    shutil.copy2(HISTORICAL_960_WEIGHTS, HIST_LOCAL)

def inspect_checkpoint(path, expected_imgsz):
    model = YOLO(str(path), task="detect")
    args = (model.ckpt or {}).get("train_args", {})
    size = args.get("imgsz")
    assert size == expected_imgsz, f"{path.name} : imgsz enregistré={size}, attendu={expected_imgsz}. Vérifier le poids choisi."
    info = dict(path=str(path), sha256=sha256_file(path), train_args=args)
    del model
    return info

weight_info = {"initial": inspect_checkpoint(BASE_LOCAL, 640)}
if MODE == "economique":
    weight_info["adapted_960_historical"] = inspect_checkpoint(HIST_LOCAL, 960)
    assert weight_info["initial"]["sha256"] != weight_info["adapted_960_historical"]["sha256"], "Les deux poids sont identiques."

## 6. Créer le protocole et les listes immuables

Le dossier de sortie dépend de l'empreinte des données, des poids, des réglages et de
l'environnement. Une modification crée une expérience distincte. Les anciens scores
0,371 et 0,982 ne sont **jamais** importés dans les calculs.

In [ ]:
DATA_DIR = WORK / "datasets"
DATA_DIR.mkdir(exist_ok=True)

def write_list(name, paths):
    p = DATA_DIR / f"{name}.txt"
    p.write_text("\n".join(map(str, paths)) + "\n")
    return str(p)

mix_train = write_list("mixed_train", sorted(replay_train + x_lists["train"]))
mix_val = write_list("mixed_val", sorted(replay_val + x_lists["val"]))
x_test = write_list("x40_test", x_lists["test"])
s_test = write_list("source_test", s_lists["test"])

def write_yaml(name, train, val, test=None):
    data = dict(path=str(WORK), train=train, val=val, names={0: "wbc"})
    if test is not None:
        data["test"] = test
    p = DATA_DIR / f"{name}.yaml"
    p.write_text(yaml.safe_dump(data, sort_keys=False))
    return str(p)

TRAIN_YAML = write_yaml("mixed_training", mix_train, mix_val)  # Pas de test dans la configuration d'entraînement.
TEST_YAMLS = {
    "x40_test": write_yaml("evaluation_x40", mix_train, mix_val, x_test),
    "source_test": write_yaml("evaluation_source", mix_train, mix_val, s_test),
}
TRAIN_COMMON = dict(epochs=EPOCHS, patience=PATIENCE, batch=TRAIN_BATCH, lr0=LR0,
                    seed=TRAIN_SEED, deterministic=True, workers=WORKERS, optimizer="auto",
                    cos_lr=False, freeze=None, single_cls=False, amp=True,
                    cache=False, rect=False, multi_scale=0.0, close_mosaic=10,
                    save=True, plots=False, verbose=True, **AUG)
EVAL_COMMON = dict(batch=EVAL_BATCH, device=DEVICE, workers=0, iou=NMS_IOU,
                   classes=[0], max_det=300, rect=True, half=False, augment=False,
                   plots=False, verbose=False, save_json=False)
PROTOCOL = dict(schema="aster-resolution-v1", mode=MODE, environment=ENV,
                weights=weight_info, inputs_manifest_sha256=digest(manifest),
                list_order_sha256=digest({k: list(map(str, v)) for k,v in used.items()}),
                replay_seed=REPLAY_SEED, replay_ratio=REPLAY_RATIO,
                train=TRAIN_COMMON, test_sizes=list(TEST_SIZES), evaluation=EVAL_COMMON,
                ap_conf=0.001, operating_conf=OPERATING_CONF)
EXPERIMENT_ID = digest(PROTOCOL)[:16]
OUT = OUTPUT_ROOT / EXPERIMENT_ID
OUT.mkdir(parents=True, exist_ok=True)
atomic_json(OUT / "protocol.json", PROTOCOL)
atomic_json(OUT / "inputs_manifest.json", manifest)
shutil.copytree(DATA_DIR, OUT / "datasets", dirs_exist_ok=True)
(OUT / "pip_freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print("Expérience :", EXPERIMENT_ID, "— résultats dans", OUT)

## 7. Adapter le modèle à 640 et, en mode contrôlé, à 960

Chaque adaptation repart du **même modèle initial à 640**, jamais du modèle déjà adapté.
`optimizer="auto"` reproduit le choix implicite du notebook original : Ultralytics peut
choisir un taux effectif différent de `lr0` ; les arguments et journaux sont sauvegardés.
L'arrêt anticipé et le choix de `best.pt` utilisent exclusivement la validation mixte.
Avec le même budget maximal, les nombres d'époques effectivement réalisés peuvent différer.

Les poids sont copiés sur Drive à chaque sauvegarde d'époque. Une exécution terminée est
réutilisée seulement avec un marqueur de fin et une empreinte valide. **Après interruption,
le notebook recommence le bras incomplet depuis la base**, dans un nouveau sous-dossier ;
il conserve les checkpoints partiels. Cela privilégie une exécution continue comparable
plutôt qu'une reprise dont l'état des générateurs aléatoires pourrait différer.

In [ ]:
def sync_training(local_dir, remote_dir):
    local_dir, remote_dir = Path(local_dir), Path(remote_dir)
    remote_dir.mkdir(parents=True, exist_ok=True)
    for relative in ("args.yaml", "results.csv", "weights/best.pt", "weights/last.pt"):
        source = local_dir / relative
        if source.is_file():
            target = remote_dir / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            tmp = target.with_name(target.name + ".tmp")
            shutil.copy2(source, tmp)
            tmp.replace(target)

def train_resolution(size):
    remote = OUT / f"train_{size}"
    remote.mkdir(exist_ok=True)
    completed = remote / "completed.json"
    if completed.is_file():
        saved = json.loads(completed.read_text())
        best = remote / saved["best_relative"]
        assert saved["experiment_id"] == EXPERIMENT_ID and sha256_file(best) == saved["sha256"]
        print("Entraînement terminé déjà disponible :", size)
        return best
    attempt = datetime.now(timezone.utc).strftime("attempt_%Y%m%dT%H%M%S_%fZ")
    destination = remote / attempt
    model = YOLO(str(BASE_LOCAL), task="detect")
    model.add_callback("on_model_save", lambda trainer: sync_training(trainer.save_dir, destination))
    started = time.monotonic()
    try:
        model.train(data=TRAIN_YAML, imgsz=size, device=DEVICE,
                    project=str(WORK / f"train_{size}"), name=attempt, exist_ok=False, **TRAIN_COMMON)
        local = Path(model.trainer.save_dir)
        sync_training(local, destination)
        best = destination / "weights/best.pt"
        assert best.is_file(), f"Entraînement sans best.pt : {size}"
        log = pd.read_csv(local / "results.csv")
        log.columns = log.columns.str.strip()
        atomic_json(completed, dict(experiment_id=EXPERIMENT_ID, train_imgsz=size,
                    best_relative=str(best.relative_to(remote)), sha256=sha256_file(best),
                    elapsed_seconds=time.monotonic()-started, epochs_ran=len(log),
                    selected_by="Ultralytics validation fitness on mixed validation only"))
        return best
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

models = {"initial_640": dict(path=BASE_LOCAL, adaptation_imgsz=None, provenance="historical_source_training")}
new_sizes = (640,) if MODE == "economique" else (640, 960)
for size in new_sizes:
    path = train_resolution(size)
    models[f"adapted_{size}"] = dict(path=path, adaptation_imgsz=size, provenance="new_controlled_run")
if MODE == "economique":
    models["adapted_960"] = dict(path=HIST_LOCAL, adaptation_imgsz=960, provenance="historical_adaptation")
atomic_json(OUT / "models.json", {name: {**info, "path": str(info["path"]),
            "sha256": sha256_file(info["path"])} for name, info in models.items()})
print("Modèles fixés avant l'évaluation du test :", list(models))

## 8. Recalculer la mAP aux deux résolutions et sur les deux domaines

Les **12 évaluations** correspondent à 3 modèles × 2 résolutions de test × 2 domaines.
La mAP est calculée avec un seuil de collecte bas (`conf=0.001`), qui permet de construire
la courbe précision-rappel. Ce n'est pas la précision à `conf=0.18`.
`iou=0.50` ci-dessous est le seuil de **NMS**, distinct de l'IoU de correspondance utilisé
pour AP@50 et AP@50–95. Tous ces réglages sont communs aux modèles.

Les anciennes valeurs de l'article peuvent différer si le logiciel, le batch ou la NMS
différaient : seules les mesures refaites ici constituent le tableau comparatif.
Le cache est propre à cette expérience et à l'empreinte du poids évalué.

In [ ]:
def evaluate_one(name, info, dataset, size):
    weight_hash = sha256_file(info["path"])
    key = f"{name}__{dataset}__test{size}"
    saved_path = OUT / "metrics" / f"{key}.json"
    signature = digest(dict(experiment_id=EXPERIMENT_ID, weights=weight_hash,
                            dataset=dataset, imgsz=size, settings=EVAL_COMMON, conf=0.001))
    if saved_path.is_file():
        saved = json.loads(saved_path.read_text())
        assert saved["signature"] == signature, f"Cache incohérent : {saved_path}"
        return saved
    model = YOLO(str(info["path"]), task="detect")
    try:
        result = model.val(data=TEST_YAMLS[dataset], split="test", imgsz=size, conf=0.001,
                           project=str(WORK / "evaluations"), name=key, exist_ok=True, **EVAL_COMMON)
        row = dict(signature=signature, model=name, provenance=info["provenance"],
                   source_train_imgsz=640, adaptation_imgsz=info["adaptation_imgsz"],
                   eval_imgsz=size, dataset=dataset, weights_sha256=weight_hash,
                   map50=float(result.box.map50), map50_95=float(result.box.map),
                   precision_at_evaluator_best_f1=float(result.box.mp),
                   recall_at_evaluator_best_f1=float(result.box.mr))
        assert all(np.isfinite(row[k]) and 0 <= row[k] <= 1 for k in ("map50", "map50_95"))
        atomic_json(saved_path, row)
        return row
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

rows=[]
for name, info in models.items():
    for dataset in TEST_YAMLS:
        for size in TEST_SIZES:
            row = evaluate_one(name, info, dataset, size)
            rows.append(row)
            print(name, dataset, size, "mAP50 =", round(row["map50"], 4))
results = pd.DataFrame(rows)
assert len(results) == 12 and not results.duplicated(["model", "dataset", "eval_imgsz"]).any()
results.to_csv(OUT / "metrics_all.csv", index=False)
for dataset in TEST_YAMLS:
    print("\n", dataset)
    display(results[results.dataset == dataset].pivot(index="model", columns="eval_imgsz", values=["map50", "map50_95"]).round(4))

## 9. Précision et rappel au point de fonctionnement historique

Sur les 104 champs prototype, un second calcul applique **conf=0.18, NMS=0.50**.
Le matching suit le script de vérification de l'article : prédictions triées par confiance,
association gloutonne à une seule boîte de référence, IoU ≥ 0.50. Les comptes sont agrégés
sur les champs (micro-précision/rappel), pas moyennés champ par champ.
Le seuil 0.18 avait été choisi pour le modèle historique : ce tableau compare un point
fixe, sans garantir qu'il soit optimal pour chaque nouveau modèle.

In [ ]:
def iou_one_to_many(box, boxes):
    left = np.maximum(box[:2], boxes[:, :2])
    right = np.minimum(box[2:], boxes[:, 2:])
    intersection = np.clip(right-left, 0, None).prod(axis=1)
    area_a = np.clip(box[2:]-box[:2], 0, None).prod()
    area_b = np.clip(boxes[:, 2:]-boxes[:, :2], 0, None).prod(axis=1)
    return intersection / np.maximum(area_a + area_b - intersection, 1e-12)

def match_boxes(predictions, scores, reference):
    used_reference = np.zeros(len(reference), dtype=bool)
    tp = 0
    for index in np.argsort(-scores):
        if len(reference) == 0:
            break
        overlaps = iou_one_to_many(predictions[index], reference)
        overlaps[used_reference] = -1
        selected = int(np.argmax(overlaps))
        if overlaps[selected] >= 0.5:
            used_reference[selected] = True
            tp += 1
    return tp, len(predictions)-tp, len(reference)-tp

def reference_boxes(path):
    with Image.open(path) as image:
        width, height = image.size
    rows = [list(map(float, line.split())) for line in label_path(path).read_text().splitlines() if line.strip()]
    boxes=[]
    for _, x,y,w,h in rows:
        boxes.append([(x-w/2)*width, (y-h/2)*height, (x+w/2)*width, (y+h/2)*height])
    return np.asarray(boxes, dtype=float).reshape(-1,4)

operating_rows=[]
for name, info in models.items():
    for size in TEST_SIZES:
        checkpoint_hash = sha256_file(info["path"])
        key = f"{name}__test{size}"
        csv_path = OUT / "operating" / f"{key}.csv"
        record_path = csv_path.with_suffix(".json")
        signature = digest(dict(experiment=EXPERIMENT_ID, weights=checkpoint_hash, size=size,
                                conf=OPERATING_CONF, nms=NMS_IOU, matching="greedy_iou_0.50_v1"))
        if record_path.is_file():
            record = json.loads(record_path.read_text())
            assert record["signature"] == signature and sha256_file(csv_path) == record["csv_sha256"]
            fields = pd.read_csv(csv_path)
        else:
            model = YOLO(str(info["path"]), task="detect")
            field_rows=[]
            try:
                for path in x_lists["test"]:
                    prediction = model.predict(str(path), imgsz=size, conf=OPERATING_CONF,
                        iou=NMS_IOU, classes=[0], max_det=300, rect=True, half=False,
                        augment=False, device=DEVICE, verbose=False, save=False)[0]
                    boxes = prediction.boxes.xyxy.cpu().numpy()
                    scores = prediction.boxes.conf.cpu().numpy()
                    tp,fp,fn = match_boxes(boxes, scores, reference_boxes(path))
                    field_rows.append(dict(model=name, eval_imgsz=size, field=path.name, tp=tp,fp=fp,fn=fn))
            finally:
                del model
                gc.collect()
                torch.cuda.empty_cache()
            fields = pd.DataFrame(field_rows)
            csv_path.parent.mkdir(parents=True, exist_ok=True)
            fields.to_csv(csv_path, index=False)
            atomic_json(record_path, dict(signature=signature, csv_sha256=sha256_file(csv_path)))
        assert len(fields) == 104 and int((fields.tp+fields.fn).sum()) == 171
        tp,fp,fn = [int(fields[k].sum()) for k in ("tp","fp","fn")]
        operating_rows.append(dict(model=name, eval_imgsz=size, tp=tp,fp=fp,fn=fn,
            precision=tp/max(tp+fp,1), recall=tp/max(tp+fn,1), f1=2*tp/max(2*tp+fp+fn,1)))
operating = pd.DataFrame(operating_rows)
operating.to_csv(OUT / "operating_point_x40.csv", index=False)
display(operating.round(4))

## 10. Comparaisons qui répondent à la question

- **Résolution au test seule :** modèle initial testé à 960 moins le même modèle testé à 640.
- **Adaptation sans augmentation de résolution :** adapté à 640 moins initial, tous deux testés à 640.
- **Adaptation à test identique :** adapté à 960 moins initial, tous deux testés à 960.
- **Résolution pendant l'adaptation :** adapté à 960 moins adapté à 640, à résolution de test fixée.
  Cette dernière comparaison est contrôlée seulement avec `MODE="controle"`.

Les écarts sont des différences absolues de mAP (0.10 = 10 points). Ils ne constituent
pas des parts causales additives et ne sont pas des tests de significativité.

In [ ]:
def comparison_table(results_frame):
    rows=[]
    for dataset in ("x40_test", "source_test"):
        table = results_frame[results_frame.dataset == dataset].set_index(["model", "eval_imgsz"])
        def value(model, size):
            return float(table.loc[(model, size), "map50"])
        definitions = [
            ("Résolution de test seule, modèle initial", "initial_640", 960, "initial_640", 640),
            ("Adaptation à 640, test fixé à 640", "adapted_640", 640, "initial_640", 640),
            ("Adaptation à 640, test fixé à 960", "adapted_640", 960, "initial_640", 960),
            ("Adaptation à 960, test fixé à 960", "adapted_960", 960, "initial_640", 960),
            ("Résolution adaptation 960 versus 640, test 640", "adapted_960", 640, "adapted_640", 640),
            ("Résolution adaptation 960 versus 640, test 960", "adapted_960", 960, "adapted_640", 960),
        ]
        for label, a, sa, b, sb in definitions:
            va,vb=value(a,sa),value(b,sb)
            rows.append(dict(dataset=dataset, comparaison=label, valeur_a=va, valeur_b=vb,
                             delta_map50=va-vb, delta_points=100*(va-vb)))
    return pd.DataFrame(rows)

contrasts = comparison_table(results)
contrasts.to_csv(OUT / "contrasts.csv", index=False)
display(contrasts.round(4))

## 11. Figure et synthèse calculées à partir des résultats

Le graphique sépare les domaines prototype et source. L'axe vertical va de 0 à 1.
Les points reliés représentent un même modèle évalué à deux résolutions ; aucune
barre d'incertitude n'est inventée à partir d'une seule seed.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
styles = {"initial_640": ("Source checkpoint — trained at 640", "#666666", "o"),
          "adapted_640": ("Adapted at 640", "#0072B2", "s"),
          "adapted_960": ("Adapted at 960", "#D55E00", "^")}
fig, axes = plt.subplots(1,2,figsize=(11,4.8), sharey=True)
for ax, dataset, title in zip(axes, ("x40_test","source_test"), ("Prototype: 104 fields / 171 boxes", "Public domain: source test split")):
    for name,(label,color,marker) in styles.items():
        data = results[(results.dataset == dataset)&(results.model == name)].sort_values("eval_imgsz")
        ax.plot(data.eval_imgsz.astype(str), data.map50, label=label, color=color, marker=marker, linewidth=1.8)
    ax.set(title=title, xlabel="Evaluation input size (imgsz)", ylim=(0,1.03))
    ax.grid(axis="y", alpha=0.2)
axes[0].set_ylabel("mAP@50")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="lower center",ncol=3,frameon=False)
fig.suptitle(f"Adaptation et résolution — mode {MODE}")
fig.tight_layout(rect=(0,0.12,1,0.94))
fig.savefig(OUT / "resolution_comparison.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT / "resolution_comparison.pdf", bbox_inches="tight")
plt.show()

prototype = results[results.dataset == "x40_test"].set_index(["model","eval_imgsz"])
def score(model,size):
    return float(prototype.loc[(model,size),"map50"])
summary = ["# Résultats du contrôle adaptation et résolution", "", f"Expérience : {EXPERIMENT_ID}",
           f"Mode : {MODE}", "", "Mesures sur les mêmes 104 champs prototype :"]
for model in models:
    summary.append(f"- {model} : test 640 = {score(model,640):.4f} ; test 960 = {score(model,960):.4f}.")
summary += ["", f"Gain du modèle initial par passage du test 640 au test 960 : {score('initial_640',960)-score('initial_640',640):+.4f}.",
    f"Gain de l'adaptation à 640, en gardant le test à 640 : {score('adapted_640',640)-score('initial_640',640):+.4f}.",
    f"Gain de l'adaptation à 960, en gardant le test à 960 : {score('adapted_960',960)-score('initial_640',960):+.4f}.", "",
    "Les métriques de l'article (0,371 et 0,982) sont des références historiques, pas des valeurs imputées ici.",
    "Un gain à taille de test identique ne peut pas être attribué au seul agrandissement à l'inférence.",
    "Une comparaison avant/après mesure la recette d'adaptation complète (données, replay, augmentation et apprentissage supplémentaire).",
    "Isoler strictement l'ajout des données prototype demanderait aussi un contrôle de poursuite d'apprentissage sur sources seules."]
if MODE == "economique":
    summary += ["Le modèle adapté à 960 est historique : ses différences avec adapté_640 ne peuvent pas être attribuées exclusivement à la résolution d'entraînement."]
else:
    summary += ["Les deux adaptations ont été relancées avec les mêmes réglages sauf imgsz ; l'arrêt anticipé peut toutefois produire des durées d'apprentissage différentes."]
summary += ["Une seule seed et une seule lame : aucune conclusion de significativité ni généralisation à d'autres patients.",
            "Ces analyses sont complémentaires et postérieures à l'évaluation d'origine ; elles ne sont pas préenregistrées."]
(OUT / "synthese.md").write_text("\n".join(summary)+"\n")
from IPython.display import Markdown
display(Markdown("\n".join(summary)))

## 12. Archiver les résultats et terminer

Les fichiers essentiels sont `metrics_all.csv`, `operating_point_x40.csv`, `contrasts.csv`,
`synthese.md`, la figure, `protocol.json`, les listes et empreintes, et les poids de chaque
nouvel entraînement. Une archive légère rassemble les tableaux et la traçabilité ; les
poids restent dans les sous-dossiers `train_640` / `train_960` sur Drive.

Après exécution, **Fichier → Enregistrer une copie dans Drive** permet de conserver aussi
les sorties visibles du notebook. Pour une nouvelle comparaison contrôlée, changer le
mode dans la première cellule puis tout relancer ; un dossier d'expérience distinct sera créé.

In [ ]:
import zipfile
required = ["metrics_all.csv", "operating_point_x40.csv", "contrasts.csv", "synthese.md",
            "protocol.json", "inputs_manifest.json", "models.json", "pip_freeze.txt",
            "resolution_comparison.png", "resolution_comparison.pdf"]
assert all((OUT / name).is_file() for name in required)
with zipfile.ZipFile(OUT / "resultats_controle_resolution.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for name in required:
        archive.write(OUT / name, arcname=name)
    for folder in ("datasets", "metrics", "operating"):
        for path in sorted((OUT / folder).rglob("*")):
            if path.is_file():
                archive.write(path, arcname=str(path.relative_to(OUT)))
atomic_json(OUT / "experiment_completed.json", dict(experiment_id=EXPERIMENT_ID,
            completed_utc=datetime.now(timezone.utc).isoformat(), ap_evaluations=len(results),
            operating_evaluations=len(operating)))
print("Terminé. Résultats et checkpoints sur Drive :", OUT)
print("Archive des tableaux et de la traçabilité :", OUT / "resultats_controle_resolution.zip")

## Lecture scientifique et limites

1. Si l'adaptation à 640 améliore le score à test 640, le gain de cette comparaison
   ne vient pas d'une augmentation de résolution. Il concerne la recette mixed-replay.
2. Si le modèle initial reste inférieur au modèle adapté à test 960, un agrandissement
   au test ne suffit pas à reproduire les performances du modèle adapté.
3. En mode contrôlé, comparer les deux adaptations à test fixe renseigne sur l'intérêt
   de l'entrée 960 pendant l'adaptation, sous cette recette et cette seed.
4. Conserver tous les résultats, y compris une baisse du domaine source ou un contrôle
   défavorable. Ne pas choisir une seed, une NMS ou un modèle en fonction du test.
5. Les 104 champs sont issus d'une seule lame. Le choix historique de la base avait déjà
   utilisé des résultats sur le prototype : il ne s'agit pas d'un nouveau test indépendant.

**Origine du protocole :** notebook local `05_mixedreplay_x40_colab.ipynb`, scripts
`finetune_x40_mixedreplay.py`, `eval_on_x40_test.py` et `run_detections_x40test.py`.
La version 8.4.53 est celle déclarée dans le manuscrit ; elle ne certifie pas la version
exacte du runtime historique. Le mode contrôlé évite de comparer deux nouvelles adaptations
faites sous des versions différentes.

Références API : [validation Ultralytics](https://docs.ultralytics.com/modes/val/),
[paramètres d'entraînement](https://docs.ultralytics.com/modes/train/),
[sauvegardes par callbacks](https://docs.ultralytics.com/usage/callbacks/).